# 🌍 Spatial-LLM — Multi-Modal Training on Google Colab

Trains the neuroscience-inspired Spatial-LLM with **three modalities**:
1. **Coordinates** (lat/lon) → BrainSpatialCortex
2. **Map tiles** (satellite images) → ViT encoder
3. **Text** (questions) → LLM backbone

**Runtime → Change runtime type → T4 GPU (free) or A100 (Pro).**

## 1. Setup — clone repo & install

In [ ]:
!git clone https://github.com/Mohammadzamanid/Spatial-LLM.git
%cd Spatial-LLM
!pip install -q torch torchvision transformers peft accelerate datasets
!pip install -q geopandas shapely mercantile pyyaml pillow
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Generate synthetic multi-modal data
Replace this with your real datasets (GeoQA, OSM, BigEarthNet) when ready.

In [ ]:
!python -m src.data.synthetic --n_train 5000 --n_val 500 --output_dir data/processed/
import json
with open('data/processed/train.jsonl') as f:
    sample = [json.loads(l) for _, l in zip(range(3), f)]
sample

## 3. Run the ablation harness on GPU
See which modules actually contribute before full training.

In [ ]:
# Leave-one-out: which modules are load-bearing?
!python -m src.eval.ablation --mode leave_one_out

# Test synchronization (per-module auxiliary losses)
!python -m src.eval.ablation --mode leave_one_out --aux_loss

## 4. Full multi-modal training (coords + tiles + text + LLM)
Uses LoRA so a 7B model fits on a single T4/A100.

In [ ]:
# Optional: log in to HF for gated models like Mistral/LLaMA
# from huggingface_hub import notebook_login; notebook_login()

!python -m src.training.trainer --config configs/train_config.yaml

## 5. Evaluate & inference

In [ ]:
!python -m src.eval.benchmark --config configs/train_config.yaml --checkpoint outputs/run_001/best

!python -m src.inference --config configs/train_config.yaml --checkpoint outputs/run_001/best \
  --lat 35.6895 --lon 139.6917 --question 'What type of urban area is this?'

## 6. Add your own modalities

To train on **real** multi-modal data:
- **Map tiles:** set `use_tiles: true` in config; tiles fetch automatically from OSM/ESRI per coordinate
- **Audio / other:** add an encoder in `src/models/` following the `SpatialTileEncoder` pattern, then concatenate its tokens in `llm_wrapper.py`
- **Movement (heading/speed):** the `ConjunctiveSpatialCells` and `PhasePrecession` modules consume these — feed real trajectory data to wake them up (the ablation showed they're dormant without it)

Save checkpoints to Google Drive so they persist across sessions:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r outputs/ /content/drive/MyDrive/spatial-llm-checkpoints/